### SAE Collecte de données web 
### Christella DAVID - Santatriniaina RASOLOFOMANANA - BUT 2AFA VCOD 

## 1. Importation des bibliothèques

Nous commençons par importer les bibliothèques nécessaires pour notre traitement de données et web scraping.

In [36]:
import pandas as pd
from selenium import webdriver
from bs4 import BeautifulSoup
import time
import requests

## 2. Chargement des données

Nous chargeons le fichier contenant les informations sur les commerces parisiens. Nous limitons à 1000 commerces pour ce traitement.

In [37]:
path = r"C:\Users\ChristellaDAVID\OneDrive - STARTUM CFA\Bureau\Stella\BDCOM_2023.xls"
bdcom = pd.read_csv(path, nrows=1000)

## 3. Extraction des adresses pour la géolocalisation

Nous extrayons les colonnes nécessaires pour construire les adresses qui seront envoyées à l'API de géolocalisation.

In [38]:
exporter_adresse = bdcom[["OBJECTID", "num", "let", "typ_voie", "lib_voie", "arro"]]
exporter_adresse.to_csv("exporter_geo.csv", index=False)

## 4. Géolocalisation avec l'API adresse.data.gouv.fr

Nous utilisons l'API publique adresse.data.gouv.fr pour convertir les adresses en coordonnées GPS (latitude et longitude).

In [39]:
url = "https://api-adresse.data.gouv.fr/search/csv/"
fichier = open("exporter_geo.csv", "rb")
r = requests.post(url, files={"data": fichier})


fichier_resultat = open("exporter_geo_sorties.csv", "w", encoding="utf-8")
fichier_resultat.write(r.text)
fichier_resultat.close()

## 5. Fusion des données géolocalisées avec les données d'origine

Nous fusionnons les résultats de géolocalisation avec notre dataframe d'origine et ajoutons les colonnes pour stocker les sites web et informations de livraison.

In [40]:

geo = pd.read_csv("exporter_geo_sorties.csv")

# Fusion
bcom_geo = bdcom.merge(geo, on="OBJECTID", how="left")

# Ajout des colonnes pour le site web et la livraison
bcom_geo["website"] = None
bcom_geo["livraison_possible"] = "Inconnu"

bcom_geo.to_csv("BDCOM2023_complet_gps.csv", index=False)

## 6. Définition de la fonction de formatage pour les adresses

Cette fonction nous permet de formater correctement les éléments d'adresse pour construire les URL de recherche Google Maps.

In [41]:
def f(adresse_element, est_dernier_element=False):
    # Initialisation
    resultat_formate = ""
    # Vérification que l'élément n'est pas une valeur manquante 
    if not(pd.isna(adresse_element)):
        # Si l'élément est un nombre décimal, le convertir en entier puis en chaîne
        if isinstance(adresse_element, float):
            resultat_formate += str(int(adresse_element))
        # Sinon, simplement convertir en chaîne
        else:
            resultat_formate += str(adresse_element)
        if not(est_dernier_element):
            resultat_formate += "+"
    return resultat_formate

## 7. Configuration du traitement par lots

Nous configurons le traitement par lots pour optimiser la collecte de données et réduire le risque d'être bloqués par Google.

In [43]:
# traitement par lots
total_commerces = len(bdcom)
taille_lot = 50
pause_entre_lots = 5
pause_entre_requetes = 0.2

# navigateur
driver = webdriver.Chrome()
base_url = "https://www.google.com/maps/search/"

# stockage des sites web
sites = [None] * total_commerces

## 8. Traitement par lots des commerces

Cette étape principale parcourt les commerces par lots, recherche leurs sites web sur Google Maps et vérifie la possibilité de livraison.

In [ ]:
# Traitement par lots
for debut_lot in range(0, total_commerces, taille_lot):
    fin_lot = min(debut_lot + taille_lot, total_commerces)

    
    for i in range(debut_lot, fin_lot):
        try:
            commerce = bdcom.iloc[i]

            place_info = ""
            if not pd.isna(commerce["ens"]):
                place_info += f(commerce["ens"])

            place_info += f(commerce["num"]) + f(commerce["let"]) + \
                         f(commerce["typ_voie"]) + f(commerce["lib_voie"]) + \
                         f(commerce["arro"], True)

            # URL complète
            url = base_url + place_info

            
            driver.get(url)
            time.sleep(pause_entre_requetes)  

            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            resultat = soup.select("a[aria-label*='Site Web']")

            # Stockage du résultat
            if resultat:
                website = resultat[0]["href"]
                sites[i] = website

                # Vérification de la livraison
                try:
                    driver.get(website)
                    time.sleep(pause_entre_requetes)
                    page_content = driver.page_source.lower()
                    # Recherche de mots liés à la livraison
                    if any(keyword in page_content for keyword in ["livraison", "delivery", 
                                                                   "uber eats", "deliveroo", "just eat"]):
                        bcom_geo.loc[i, "livraison_possible"] = "Oui"
                    else:
                        bcom_geo.loc[i, "livraison_possible"] = "Non"
                except:
                    pass
        except Exception as e:
            pass

    # Mise à jour des données de sortie 
    for i in range(debut_lot, fin_lot):
        if sites[i] is not None:
            bcom_geo.loc[i, "website"] = sites[i]

    # Sauvegarde 
    bcom_geo.to_csv(f"BDCOM2023_mise_a_jour{debut_lot+1}-{fin_lot}.csv", index=False)

    # Pause entre les lots 
    if fin_lot < total_commerces: 
        time.sleep(pause_entre_lots)

## 9. Finalisation et sauvegarde des résultats

Nous fermons le navigateur et sauvegardons les résultats finaux dans un fichier CSV. Cette étape conclut notre processus d'enrichissement des données.

In [ ]:
# Fermeture du navigateur
driver.close()

# base de donnée finale
bcom_geo.to_csv("BDCOM2023_final.csv", index=False)